In [1]:
!pip install langchain
!pip install langchain-core
!pip install langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49

In [23]:
from langchain_nomic.embeddings import NomicEmbeddings
from google.colab import userdata
nomic_api_key = userdata.get('nomic_api_key')
embedding = NomicEmbeddings(model="nomic-embed-text-v1.5",nomic_api_key= nomic_api_key,inference_mode="local")

In [4]:
!pip install langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: aiohttp
    Found existing installation: aiohttp 3.10.8
    Uninstalling aiohttp-3.10.8:
      Successfully uninstalled aiohttp-3.10.8


In [1]:
from google.colab import userdata
pinecone_api_key = userdata.get('pinecone_api_key')

In [2]:
from pinecone import Pinecone, ServerlessSpec
import time
# from config import VECTOR_DIMENSION
VECTOR_DIMENSION = 384 # 768 do if nomic embeddings using
class PineconeManager:
    def __init__(self, api_key: str, index_name: str):
        self.pc = Pinecone(api_key=api_key)
        self.index_name = index_name
        self.index = None
        self.initialize_index() #to initialize the index
    def initialize_index(self):
        if self.index_name not in self.pc.list_indexes().names():#shd use list indexes here
            print(f"Creating index: {self.index_name}")
            self.pc.create_index(
                name=self.index_name,
                dimension=VECTOR_DIMENSION,
                metric="cosine",
                spec=ServerlessSpec(
                    cloud="aws",
                    region="us-east-1"
                )
            )
        else:
            print(f"Index {self.index_name} already exists")

        while not self.pc.describe_index(self.index_name).status['ready']:
            time.sleep(1)

        self.index = self.pc.Index(self.index_name)


In [3]:
INDEX_NAME = 'dlprojectcheck'
pinecone_manager = PineconeManager(pinecone_api_key, INDEX_NAME)
pinecone_manager.initialize_index()

Creating index: dlprojectcheck
Index dlprojectcheck already exists


In [4]:
import os
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from typing import List, Dict
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

class WebScraper:
    def __init__(self):
        pass

    @staticmethod
    async def extract_sections(soup: BeautifulSoup) -> List[Dict[str, str]]:
        """Extract sections from a parsed HTML page.

        Sections are defined as the contents between headers (h1, h2, h3) and are
        dictionaries with keys "title", "content", and "code". The value of "title"
        is the text of the header, "content" is the text content of the section, and
        "code" is the code blocks in the section.

        :param soup: A BeautifulSoup object representing the HTML page
        :return: A list of section dictionaries
        """
        sections = []
        current_section = {"title": "Introduction", "content": "", "code": ""}

        for element in soup.find_all(['h1', 'h2', 'h3', 'p', 'pre']):
            if element.name in ['h1', 'h2', 'h3']:
                if current_section["content"] or current_section["code"]:
                    sections.append(current_section)
                current_section = {"title": element.get_text(strip=True), "content": "", "code": ""}
            elif element.name == 'p':
                current_section["content"] += element.get_text(strip=True) + "\n"
            elif element.name == 'pre':
                code = element.find('code')
                if code:
                    current_section["code"] += code.get_text(strip=True) + "\n\n"

        if current_section["content"] or current_section["code"]:
            sections.append(current_section)

        return sections

    def create_documents(self, sections: List[Dict[str, str]], file_path: str) -> List[Document]:
        """Create a list of Document objects from a list of sections and a file path.

        The content of each section is combined with its code block (if any) and
        used to create a Document object. The metadata of the Document includes
        the title of the section and the source file path.

        :param sections: A list of dictionaries with keys "title", "content", and "code"
        :param file_path: The path to the file containing the sections
        :return: A list of Document objects
        """
        documents = []
        for section in sections:
            content = section["content"]
            code = section["code"]
            combined_text = f"{content}\n\nCode:\n{code}"

            document = Document(
                page_content=combined_text,
                metadata={
                    "title": section["title"],
                    "source": file_path
                }
            )
            documents.append(document)

        return documents

    async def scrape_file(self, file_path: str) -> List[Document]:
        """Scrape a file and return a list of Document objects.

        The file is read and parsed with BeautifulSoup, and then the sections
        are extracted and combined into Document objects. The metadata of each
        Document includes the title of the section and the source file path.

        :param file_path: The path to the file to scrape
        :return: A list of Document objects
        """
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                html = file.read()
                soup = BeautifulSoup(html, 'html.parser')
                sections = await self.extract_sections(soup)

                documents = self.create_documents(sections, file_path)
                return documents
        except Exception as e:
            print(f"Error scraping {file_path}: {str(e)}")
            return []

    async def scrape_files(self, file_paths: List[str]) -> List[Document]:
        """Scrape a list of files and return a list of Document objects.

        This function will scrape each file in the list in parallel using asyncio.
        The documents from each file are collected and returned as a single list.

        :param file_paths: A list of file paths to scrape
        :return: A list of Document objects
        """
        tasks = [self.scrape_file(file_path) for file_path in file_paths]
        documents_list = await asyncio.gather(*tasks)
        all_documents = [doc for sublist in documents_list for doc in sublist]
        return all_documents


In [5]:
from google.colab import drive
import os

# Mount your Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
scraper = WebScraper()
dir_to_work_with = '/content/drive/MyDrive/api'
html_files = []
for root, dirs, files in os.walk(dir_to_work_with):
    for file in files:
        if file.endswith(".html"):
            html_files.append(os.path.join(root, file))
print("the html files got from directory and embedding initialized\n")

print("document scrape started")
documents = await scraper.scrape_files(file_paths=html_files)
print("documents scrape done \n")

the html files got from directory and embedding initialized

document scrape started
documents scrape done 



In [7]:
documents

[Document(metadata={'title': 'What is mixed precision training?', 'source': '/content/drive/MyDrive/api/mixed_precision.html'}, page_content='Mixed precision training is the use of lower-precision operations (float16andbfloat16) in a model\nduring training to make it run faster and use less memory.\nUsing mixed precision can improve performance by more than 3 times on modern GPUs and 60% on TPUs.\nToday, most models use thefloat32dtype, which takes 32 bits of memory.\nHowever, there are two lower-precision dtypes,float16andbfloat16,\neach which take 16 bits of memory instead. Modern accelerators like Google TPUs and NVIDIA GPUs \ncan run operations faster in the 16-bit dtypes,\nas they have specialized hardware to run 16-bit computations and 16-bit dtypes can be read from memory faster.\nTherefore, these lower-precision dtypes should be used whenever possible on those devices.\nHowever, variables storage (as well as certain sensitive computations) should still be infloat32to preserve n

In [9]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 5.5 MB/s eta 0:00:00


In [10]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=pinecone_manager.index, embedding=embeddings)

vector_store.add_documents(documents)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

['7ca6ccf7-4ec9-476d-9467-757641226727',
 '905f8aa5-dee0-4c12-ba92-4a8330515992',
 '4ab7fa39-fc8f-479f-8917-803a510a7a00',
 'c83a1e49-e9ac-4fd1-b12f-ed38bd5f94b5',
 'c8d9357c-1749-4e6f-bda8-eb9451f816ca',
 '5aa67e0e-8f32-44af-a715-bd5fac5f41fc',
 '02d98edf-c5f4-4004-9f98-7bb8648b46e4',
 'af809da6-c6f7-4c4c-9d67-96ff75a64582',
 'de3f445e-fa8d-4646-aa26-b2e685a869b9',
 '70fdd797-23a6-485c-8573-45422a14f07d',
 'e76b95bf-8231-4c60-acb1-06e2a20e044c',
 '12151c98-93f2-4b77-8bd6-02e9518c2ff3',
 'f92430a8-a53e-46cf-9bba-ae4e1cab8620',
 '321feba4-094f-47f2-b7ab-53be2c1d365b',
 'a916ea43-fde9-4877-ab82-b5024c169978',
 '056ef6f0-9c04-4c48-b286-2738696b04ce',
 'b5c2d27a-8aba-4da4-874b-8cfde905aff2',
 '59e44363-15d7-48de-952c-999c148df700',
 'f7c86b10-d461-4c55-b89e-a7e234bea314',
 '3b5c991d-54d5-4b93-a31a-1b5a9ab54d1e',
 '3048d740-726e-4155-9603-6e8ca9aec355',
 'ba16033f-9402-40ff-9ac1-22e7e4e1cde5',
 '6a004ca9-5f20-43d7-9d33-627c96c41b67',
 '9f53011f-61e1-4a7d-81da-f8378740019f',
 '267cee66-66e1-

In [11]:
query = "How does keras handle Layers?"
vector_store.similarity_search_with_relevance_scores(query)

[(Document(id='a047be56-cfdc-41ab-b5b9-55663a5a4fd0', metadata={'source': '/content/drive/MyDrive/api/layers.html', 'title': 'Keras layers API'}, page_content="Layers are the basic building blocks of neural networks in Keras.\nA layer consists of a tensor-in tensor-out computation function (the layer'scallmethod)\nand some state, held in TensorFlow variables (the layer'sweights).\nA Layer instance is callable, much like a function:\nUnlike a function, though, layers maintain a state, updated when the layer receives data\nduring training, and stored inlayer.weights:\n\n\nCode:\nfromtensorflow.kerasimportlayerslayer=layers.Dense(32,activation='relu')inputs=tf.random.uniform(shape=(10,20))outputs=layer(inputs)\n\n>>>layer.weights[<tf.Variable'dense/kernel:0'shape=(20,32)dtype=float32>,<tf.Variable'dense/bias:0'shape=(32,)dtype=float32>]\n\n"),
  0.8609783055),
 (Document(id='7ab070f3-978a-4b53-88f9-95171f5acb06', metadata={'source': '/content/drive/MyDrive/api/layers/index.html', 'title':

In [15]:
vector_store.similarity_search(query)

[Document(id='7ab070f3-978a-4b53-88f9-95171f5acb06', metadata={'source': '/content/drive/MyDrive/api/layers/index.html', 'title': 'Keras layers API'}, page_content="Layers are the basic building blocks of neural networks in Keras.\nA layer consists of a tensor-in tensor-out computation function (the layer'scallmethod)\nand some state, held in TensorFlow variables (the layer'sweights).\nA Layer instance is callable, much like a function:\nUnlike a function, though, layers maintain a state, updated when the layer receives data\nduring training, and stored inlayer.weights:\n\n\nCode:\nfromtensorflow.kerasimportlayerslayer=layers.Dense(32,activation='relu')inputs=tf.random.uniform(shape=(10,20))outputs=layer(inputs)\n\n>>>layer.weights[<tf.Variable'dense/kernel:0'shape=(20,32)dtype=float32>,<tf.Variable'dense/bias:0'shape=(32,)dtype=float32>]\n\n"),
 Document(id='a047be56-cfdc-41ab-b5b9-55663a5a4fd0', metadata={'source': '/content/drive/MyDrive/api/layers.html', 'title': 'Keras layers API'

In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever.get_relevant_documents(query)

<ipython-input-16-66b0662f0e42>:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retriever.get_relevant_documents(query)


[Document(id='a047be56-cfdc-41ab-b5b9-55663a5a4fd0', metadata={'source': '/content/drive/MyDrive/api/layers.html', 'title': 'Keras layers API'}, page_content="Layers are the basic building blocks of neural networks in Keras.\nA layer consists of a tensor-in tensor-out computation function (the layer'scallmethod)\nand some state, held in TensorFlow variables (the layer'sweights).\nA Layer instance is callable, much like a function:\nUnlike a function, though, layers maintain a state, updated when the layer receives data\nduring training, and stored inlayer.weights:\n\n\nCode:\nfromtensorflow.kerasimportlayerslayer=layers.Dense(32,activation='relu')inputs=tf.random.uniform(shape=(10,20))outputs=layer(inputs)\n\n>>>layer.weights[<tf.Variable'dense/kernel:0'shape=(20,32)dtype=float32>,<tf.Variable'dense/bias:0'shape=(32,)dtype=float32>]\n\n"),
 Document(id='7ab070f3-978a-4b53-88f9-95171f5acb06', metadata={'source': '/content/drive/MyDrive/api/layers/index.html', 'title': 'Keras layers API'

In [12]:
pinecone_manager.index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 1156}},
 'total_vector_count': 1156}

In [13]:
pinecone_manager.index

In [14]:
pinecone_manager.index_name
#very imp once the index is done then using that only accessing it so it is important

'dlprojectcheck'

from here for the nomic part to be done

In [24]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=pinecone_manager.index, embedding=embedding)

In [24]:
!pip install langchain-nomic "nomic[local]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00


In [37]:
!pip install "nomic[local]"

In [19]:
import torch

# Check if CUDA (GPU) is available
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)  # Get the name of the first GPU
    print(f"Using GPU: {device_name}")
else:
    print("Using CPU")


Using GPU: Tesla T4


In [ ]:
#want to see how many have been added how to do that
vector_store.add_documents(documents)

In [ ]:
#use of all these with nomic embeddings in lab try

##the pinecone initialization vector dimensions shd be changed so remember that
from langchain_nomic.embeddings import NomicEmbeddings
from google.colab import userdata
nomic_api_key = userdata.get('nomic_api_key')
embedding = NomicEmbeddings(model="nomic-embed-text-v1.5",nomic_api_key= nomic_api_key,inference_mode="local")

from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=pinecone_manager.index, embedding=embedding)

scraper = WebScraper()
dir_to_work_with = '/content/drive/MyDrive/api'
html_files = []
for root, dirs, files in os.walk(dir_to_work_with):
    for file in files:
        if file.endswith(".html"):
            html_files.append(os.path.join(root, file))
print("the html files got from directory and embedding initialized\n")

print("document scrape started")
documents = await scraper.scrape_files(file_paths=html_files)
print("documents scrape done \n")

#want to see how many have been added how to do that
vector_store.add_documents(documents)